# Testing the PHEE MVP Schema Regex
This notebook demonstrates and tests the regex generation for the 'main' event extraction format.

In [1]:
import re
import sys
import os

# Ensure we can import from the project root and current directory
sys.path.append(os.path.abspath(".."))
from schema import get_schema_main
from splitter import find_valid_phrases_list
from helper import EVENT_TYPES, ORDER_MAIN

print(f"Event Types: {EVENT_TYPES}")
print(f"Order Main: {ORDER_MAIN}")

Event Types: ['Adverse_event', 'Potential_therapeutic_event']
Order Main: ['event_type', 'subject', 'treatment', 'effect']


In [2]:
# Sample text for testing
text = "The patient experienced acute liver failure after starting amiodarone therapy."

# Extract valid phrases from the text
valid_phrases = find_valid_phrases_list(text)
print(f"Number of valid phrases: {len(valid_phrases)}")

# Generate the regex pattern
regex_pattern = get_schema_main(valid_phrases)
print("\nGenerated Regex Pattern (truncated):")
print(regex_pattern[:200] + "...")

# Compile for testing
pattern = re.compile(regex_pattern)

Number of valid phrases: 65

Generated Regex Pattern (truncated):
\[(\((['\"](Adverse_event|Potential_therapeutic_event)['\"]), ((['\"]null['\"])|(['\"](The|The patient|The patient experienced|The patient experienced acute|The patient experienced acute liver|The pat...


In [4]:
# Test cases
test_cases = [
    # 1. Valid: Single event with phrases from text (single quotes)
    "[('Adverse_event', 'The patient', 'amiodarone therapy', 'acute liver failure')]",
    
    # 2. Valid: Multiple events (mixed single and double quotes)
    "[(\"Adverse_event\", 'The patient', 'amiodarone', 'acute liver failure'), ('Potential_therapeutic_event', 'null', 'therapy', 'null')]",
    
    # 3. Valid: Mixed phrases and null (double quotes for null)
    "[('Adverse_event', \"null\", 'amiodarone', 'acute liver failure')]",
    
    # 4. Valid: Empty list (if allow_empty=True)
    "[]",
    
    # 5. Invalid: Wrong event type
    "[('Normal_event', 'The patient', 'amiodarone', 'null')]",
    
    # 6. Invalid: Phrase not in text
    "[('Adverse_event', 'The patient', 'aspirin', 'headache')]"
]

# Re-generate and compile to ensure we use the latest function version
valid_phrases = find_valid_phrases_list(text)
regex_pattern = get_schema_main(valid_phrases)
pattern = re.compile(regex_pattern)

print("Regex Matching Results:\n")
for i, test in enumerate(test_cases, 1):
    is_match = bool(pattern.fullmatch(test.strip()))
    status = "✅ MATCH" if is_match else "❌ NO MATCH"
    print(f"Case {i}: {status}\n   Input: {test}\n")

Regex Matching Results:

Case 1: ✅ MATCH
   Input: [('Adverse_event', 'The patient', 'amiodarone therapy', 'acute liver failure')]

Case 2: ✅ MATCH
   Input: [("Adverse_event", 'The patient', 'amiodarone', 'acute liver failure'), ('Potential_therapeutic_event', 'null', 'therapy', 'null')]

Case 3: ✅ MATCH
   Input: [('Adverse_event', "null", 'amiodarone', 'acute liver failure')]

Case 4: ✅ MATCH
   Input: []

Case 5: ❌ NO MATCH
   Input: [('Normal_event', 'The patient', 'amiodarone', 'null')]

Case 6: ❌ NO MATCH
   Input: [('Adverse_event', 'The patient', 'aspirin', 'headache')]



## Testing get_schema_sub
Testing the regex for the detailed dictionary format.

In [3]:
from schema import get_schema_sub
from helper import SUBJECT_FIELDS, TREATMENT_FIELDS

# Generate the sub regex pattern
regex_pattern_sub = get_schema_sub(valid_phrases)
pattern_sub = re.compile(regex_pattern_sub)

# Test cases for SUB format
test_cases_sub = [
    # 1. Valid: Single event dictionary (Double quotes for keys, single/double for values)
    '[{"event_type": "Adverse_event", "subject": ("The patient", "null", "null", "null", "null", "null"), "treatment": ("amiodarone therapy", "amiodarone", "null", "null", "null", "null", "null", "null", "null"), "effect": "acute liver failure"}]',
    
    # 2. Valid: Multiple dictionaries
    '[{"event_type": "Adverse_event", "subject": ("The patient", "null", "null", "null", "null", "null"), "treatment": ("amiodarone", "amiodarone", "null", "null", "null", "null", "null", "null", "null"), "effect": "liver failure"}, {"event_type": "Potential_therapeutic_event", "subject": ("null", "null", "null", "null", "null", "null"), "treatment": ("therapy", "null", "null", "null", "null", "null", "null", "null", "null"), "effect": "null"}]',
    
    # 3. Invalid: Wrong number of tuple elements in subject
    '[{"event_type": "Adverse_event", "subject": ("The patient", "null"), "treatment": ("null", "null", "null", "null", "null", "null", "null", "null", "null"), "effect": "null"}]',
    
    # 4. Invalid: Missing key
    '[{"event_type": "Adverse_event", "subject": ("null", "null", "null", "null", "null", "null"), "treatment": ("null", "null", "null", "null", "null", "null", "null", "null", "null")}]',

    # 5. Invalid: Single quotes for keys (JSON keys must be double quoted)
    "[{'event_type': 'Adverse_event', 'subject': ('null', 'null', 'null', 'null', 'null', 'null'), 'treatment': ('null', 'null', 'null', 'null', 'null', 'null', 'null', 'null', 'null'), 'effect': 'null'}]"
]

print("Regex Matching Results (SUB):\n")
for i, test in enumerate(test_cases_sub, 1):
    is_match = bool(pattern_sub.fullmatch(test.strip()))
    status = "✅ MATCH" if is_match else "❌ NO MATCH"
    print(f"Case {i}: {status}\n   Input: {test}\n")

Regex Matching Results (SUB):

Case 1: ✅ MATCH
   Input: [{"event_type": "Adverse_event", "subject": ("The patient", "null", "null", "null", "null", "null"), "treatment": ("amiodarone therapy", "amiodarone", "null", "null", "null", "null", "null", "null", "null"), "effect": "acute liver failure"}]

Case 2: ✅ MATCH
   Input: [{"event_type": "Adverse_event", "subject": ("The patient", "null", "null", "null", "null", "null"), "treatment": ("amiodarone", "amiodarone", "null", "null", "null", "null", "null", "null", "null"), "effect": "liver failure"}, {"event_type": "Potential_therapeutic_event", "subject": ("null", "null", "null", "null", "null", "null"), "treatment": ("therapy", "null", "null", "null", "null", "null", "null", "null", "null"), "effect": "null"}]

Case 3: ❌ NO MATCH
   Input: [{"event_type": "Adverse_event", "subject": ("The patient", "null"), "treatment": ("null", "null", "null", "null", "null", "null", "null", "null", "null"), "effect": "null"}]

Case 4: ❌ NO MATCH
   In